<a href="https://colab.research.google.com/github/raki-rankawat/thesis-v1/blob/master/Pipeline_Quantz_v2_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline_Quantz — Both Seeds
Exports **`mobilenetv2_seed_74.pth`** and **`mobilenetv3_seed_74.pth`** to:
- FP32 ONNX
- INT8 QDQ ONNX (static quantisation, calibrated on train split)

Each model gets its own output directory under `exports/`.  
Test split is used for on-device evaluation accuracy — never for calibration.

Run all cells top-to-bottom; no manual config changes needed between models.

In [1]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

Mounted at /content/drive
✅ utils loaded from Drive


In [2]:
# ── Imports ──────────────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path

!pip -q install onnx onnxruntime onnxruntime-tools
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat, CalibrationDataReader

from utils.dataset import prepare_dataset, get_loaders, get_test_loader
from utils.models  import VWW_MobileNetV2, VWW_MobileNetV3
from utils.train   import setup_device, evaluate

device = setup_device(seed=41)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.2 MB/s eta 0:00:00
Device: cuda


In [3]:
# ── Config ───────────────────────────────────────────────────────────
SAVE_DIR  = "/content/drive/My Drive/stm32-thesis/checkpoints"
ROOTS_DIR = Path("/content/drive/My Drive/stm32-thesis/exports")

# Both checkpoints to process — order doesn't matter
RUNS = [
    {
        "ckpt"      : f"{SAVE_DIR}/mobilenetv2_seed_74.pth",
        "arch"      : "mobilenetv2",
        "out_dir"   : ROOTS_DIR / "mobilenetv2_seed_74",
    },
    {
        "ckpt"      : f"{SAVE_DIR}/mobilenetv3_seed_74.pth",
        "arch"      : "mobilenetv3",
        "out_dir"   : ROOTS_DIR / "mobilenetv3_seed_74",
    },
]

for r in RUNS:
    r["out_dir"].mkdir(parents=True, exist_ok=True)

print("Output dirs:")
for r in RUNS:
    exists = "✅" if Path(r["ckpt"]).exists() else "❌ MISSING"
    print(f"  {r['arch']:14s}  ckpt: {exists}   out: {r['out_dir']}")

Output dirs:
  mobilenetv2     ckpt: ✅   out: /content/drive/My Drive/stm32-thesis/exports/mobilenetv2_seed_74
  mobilenetv3     ckpt: ✅   out: /content/drive/My Drive/stm32-thesis/exports/mobilenetv3_seed_74


In [4]:
# ── Export helpers ───────────────────────────────────────────────────

def build_model(arch: str) -> nn.Module:
    """Instantiate the right architecture from the arch string."""
    if arch == "mobilenetv2":
        return VWW_MobileNetV2().to(device)
    elif arch == "mobilenetv3":
        return VWW_MobileNetV3().to(device)
    else:
        raise ValueError(f"Unknown arch: {arch!r}. Expected 'mobilenetv2' or 'mobilenetv3'.")


def export_onnx(model: nn.Module, path: Path) -> None:
    model.eval()
    dummy = torch.randn(1, 3, 96, 96, device=device)
    torch.onnx.export(
        model, dummy, str(path),
        input_names=["input"], output_names=["logits"],
        export_params=True, opset_version=18,
        do_constant_folding=True,
        dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
        dynamo=False,
    )
    onnx.checker.check_model(str(path), full_check=False)
    print(f"    ✅ FP32 ONNX: {path}")


def save_test_files(model: nn.Module, test_loader, out_dir: Path, n: int = 200):
    """Save input / logits / labels from TEST split for STM32 on-device eval."""
    inputs, logits_list, labels = [], [], []
    model.eval()
    with torch.no_grad():
        for i, (x, y) in enumerate(test_loader):
            if i >= n:
                break
            x = x.to(device)
            out = model(x)
            inputs.append(x.cpu().numpy().astype("float32")[0])
            logits_list.append(out.cpu().numpy().astype("float32")[0])
            labels.append(int(y.item()))
    inputs = np.stack(inputs)
    labels = np.array(labels, dtype="int32")
    logits = np.stack(logits_list)
    np.savez(out_dir / "test_input.npz",  input=inputs)
    np.savez(out_dir / "test_labels.npz", label=labels)
    np.savez(out_dir / "test_logits.npz", input=inputs, logits=logits)
    print(f"    ✅ Test files saved — inputs: {inputs.shape}  labels: {labels.shape}")
    return out_dir / "test_input.npz", out_dir / "test_labels.npz"


def save_calib_file(train_loader, path: Path, n: int = 200) -> None:
    """Calibration data from TRAIN split only — never test or val."""
    xs = []
    with torch.no_grad():
        for i, (x, _) in enumerate(train_loader):
            if i >= n:
                break
            xs.append(x.numpy().astype("float32")[0])
    xs = np.stack(xs)
    np.savez(path, input=xs)
    print(f"    ✅ Calib data saved: {path}  {xs.shape}")


class CalibReader(CalibrationDataReader):
    def __init__(self, npz_path):
        self.data = np.load(npz_path)["input"].astype("float32")
        self.i = 0
    def get_next(self):
        if self.i >= len(self.data):
            return None
        out = {"input": self.data[self.i : self.i + 1]}
        self.i += 1
        return out
    def rewind(self):
        self.i = 0


def quantize_int8(fp32_path: Path, calib_path: Path, int8_path: Path) -> None:
    quantize_static(
        model_input=str(fp32_path),
        model_output=str(int8_path),
        calibration_data_reader=CalibReader(calib_path),
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QInt8,
        weight_type=QuantType.QInt8,
        per_channel=True,
    )
    print(f"    ✅ INT8 QDQ ONNX: {int8_path}")


def onnx_sanity_check(fp32_path: Path, input_npz: Path) -> None:
    sess   = ort.InferenceSession(str(fp32_path), providers=["CPUExecutionProvider"])
    sample = np.load(input_npz)["input"][0:1]
    out    = sess.run(["logits"], {"input": sample})[0]
    print(f"    ✅ ONNX sanity — output shape: {out.shape}  logits: {out[0]}")


def compute_stm32_accuracy(
    labels_npz: Path,
    outputs_npz: Path,
    key: str = "c_outputs_1",
    num_classes: int = 2,
) -> float:
    labels = np.load(labels_npz)["label"].astype("int64")
    raw    = np.load(outputs_npz)[key]
    if raw.size != len(labels) * num_classes:
        raise ValueError(f"Size mismatch: {raw.size} vs {len(labels) * num_classes}")
    preds = np.argmax(raw.reshape(len(labels), num_classes), axis=1)
    return float((preds == labels).mean() * 100)

In [5]:
# ── Dataset (loaded once, shared across both runs) ───────────────────
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64)
test_loader              = get_test_loader(batch_size=1)
print("✅ Dataset ready")

1/4 Download
⬇️  Downloading VWW archive...
✅ Download complete: /content/vww_work/vw_coco2014_96.tar.gz
2/4 Extract
📦 Extracting VWW archive...
✅ Extraction complete: /content/vww_work/extracted
3/4 Find root
   Root: /content/vww_work/extracted/vw_coco2014_96
4/4 Manifests
✅ Manifests already exist: /content/drive/My Drive/vww_fixed_split_manifests
Train: 7000 | Val: 1500 | Batch: 64
Test: 1500 samples  ⚠️  Use only for final evaluation
✅ Dataset ready


In [6]:
# ── Main export loop — runs both models automatically ────────────────
results = []  # collected for the summary table at the end

for run in RUNS:
    arch    = run["arch"]
    ckpt    = run["ckpt"]
    out_dir = run["out_dir"]

    print(f"\n{'='*60}")
    print(f"  {arch.upper()}  |  seed 74")
    print(f"{'='*60}")

    # --- load model --------------------------------------------------
    print("[1/5] Loading model...")
    model = build_model(arch)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()
    val_acc = evaluate(model, val_loader, device)
    print(f"    Val accuracy: {val_acc * 100:.2f}%")

    # --- FP32 ONNX ---------------------------------------------------
    print("[2/5] Exporting FP32 ONNX...")
    fp32_path = out_dir / "model_fp32.onnx"
    export_onnx(model, fp32_path)

    # --- test files (TEST split) -------------------------------------
    print("[3/5] Saving test files (test split)...")
    input_npz, labels_npz = save_test_files(model, test_loader, out_dir, n=200)

    # --- calibration data (TRAIN split only) -------------------------
    print("[4/5] Saving calibration data (train split)...")
    calib_npz = out_dir / "calib_train.npz"
    save_calib_file(train_loader, calib_npz, n=200)

    # --- INT8 QDQ quantisation ---------------------------------------
    print("[5/5] Quantising to INT8 QDQ...")
    int8_path = out_dir / "model_int8_qdq.onnx"
    quantize_int8(fp32_path, calib_npz, int8_path)

    # --- ONNX runtime sanity check -----------------------------------
    onnx_sanity_check(fp32_path, input_npz)

    results.append({
        "arch"       : arch,
        "val_acc"    : val_acc * 100,
        "fp32_path"  : fp32_path,
        "int8_path"  : int8_path,
        "input_npz"  : input_npz,
        "labels_npz" : labels_npz,
        "out_dir"    : out_dir,
    })
    print(f"  ✅ {arch} done.")

print(f"\n{'='*60}")
print("  All exports complete.")
print(f"{'='*60}")


  MOBILENETV2  |  seed 74
[1/5] Loading model...
    Val accuracy: 78.40%
[2/5] Exporting FP32 ONNX...


/tmp/ipykernel_620/1749832713.py:16: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


    ✅ FP32 ONNX: /content/drive/My Drive/stm32-thesis/exports/mobilenetv2_seed_74/model_fp32.onnx
[3/5] Saving test files (test split)...
    ✅ Test files saved — inputs: (200, 3, 96, 96)  labels: (200,)
[4/5] Saving calibration data (train split)...


    ✅ Calib data saved: /content/drive/My Drive/stm32-thesis/exports/mobilenetv2_seed_74/calib_train.npz  (110, 3, 96, 96)
[5/5] Quantising to INT8 QDQ...


    ✅ INT8 QDQ ONNX: /content/drive/My Drive/stm32-thesis/exports/mobilenetv2_seed_74/model_int8_qdq.onnx
    ✅ ONNX sanity — output shape: (1, 2)  logits: [ 0.36212936 -0.38945207]
  ✅ mobilenetv2 done.

  MOBILENETV3  |  seed 74
[1/5] Loading model...
    Val accuracy: 79.13%
[2/5] Exporting FP32 ONNX...
    ✅ FP32 ONNX: /content/drive/My Drive/stm32-thesis/exports/mobilenetv3_seed_74/model_fp32.onnx
[3/5] Saving test files (test split)...
    ✅ Test files saved — inputs: (200, 3, 96, 96)  labels: (200,)
[4/5] Saving calibration data (train split)...


    ✅ Calib data saved: /content/drive/My Drive/stm32-thesis/exports/mobilenetv3_seed_74/calib_train.npz  (110, 3, 96, 96)
[5/5] Quantising to INT8 QDQ...


    ✅ INT8 QDQ ONNX: /content/drive/My Drive/stm32-thesis/exports/mobilenetv3_seed_74/model_int8_qdq.onnx
    ✅ ONNX sanity — output shape: (1, 2)  logits: [ 0.33153304 -0.32371467]
  ✅ mobilenetv3 done.

  All exports complete.


In [7]:
# =====================================================================
# HOW TO USE ON STM32
# =====================================================================
#
# For EACH model (mobilenetv2_seed_74 / mobilenetv3_seed_74):
#
#   FP32 STM32 run:
#     model    -> exports/<arch>/model_fp32.onnx
#     valinput -> exports/<arch>/test_input.npz
#
#   INT8 STM32 run:
#     model    -> exports/<arch>/model_int8_qdq.onnx
#     valinput -> exports/<arch>/test_input.npz
#
# After STM32 AI CLI run, copy output files back:
#   FP32: cp network_val_io.npz exports/<arch>/output_fp32.npz
#   INT8: cp network_val_io.npz exports/<arch>/output_int8.npz
#
# Then run the accuracy cell below.
# =====================================================================

In [22]:
# ── STM32 on-device accuracy (run after STM32 AI CLI outputs are back) ──
#
# This cell expects:
#   exports/<arch>/output_fp32.npz
#   exports/<arch>/output_int8.npz
#
# Rename / copy the STM32 CLI output files to those paths before running.

print(f"{'Arch':<18} {'Val Acc':>8} {'FP32 STM32 (200)':>12} {'INT8 STM32 (200)':>12} {'Delta INT8':>11}")
print("-" * 74)

for r in results:
    out_dir    = r["out_dir"]
    labels_npz = r["labels_npz"]
    fp32_out   = out_dir / "output_fp32.npz"
    int8_out   = out_dir / "output_int8.npz"

    fp32_acc = compute_stm32_accuracy(labels_npz, fp32_out) if fp32_out.exists() else float("nan")
    int8_acc = compute_stm32_accuracy(labels_npz, int8_out) if int8_out.exists() else float("nan")
    delta    = int8_acc - fp32_acc

    fp32_str  = f"{fp32_acc:.2f}%" if not (fp32_acc != fp32_acc) else "pending"
    int8_str  = f"{int8_acc:.2f}%" if not (int8_acc != int8_acc) else "pending"
    delta_str = f"{delta:+.2f}%"  if not (delta != delta)    else "   —"

    print(f"{r['arch']:<18} {r['val_acc']:>7.2f}% {fp32_str:>12} {int8_str:>12} {delta_str:>11}")

Arch                Val Acc FP32 STM32 (200) INT8 STM32 (200)  Delta INT8
--------------------------------------------------------------------------
mobilenetv2          78.40%       84.50%       85.50%      +1.00%
mobilenetv3          79.13%      pending       86.50%           —
